### Notebook to demonstrate FTMS DINO AutoML Training with Local Storage and Airgapped Workflow

Transfer learning is the process of transferring learned features from one application to another. It is a commonly used training technique where you use a model trained on one task and re-train to use it on a different task. Train Adapt Optimize (TAO) Toolkit is a simple and easy-to-use Python based AI toolkit for taking purpose-built AI models and customizing them with users' own data.

---

### FTMS AutoML with DINO (DEtection TRansformer with Improved deNoising anchOr boxes)

This notebook demonstrates how to use DINO object detection model with AutoML capabilities in a local storage environment designed for airgapped workflows.

[DINO](https://arxiv.org/abs/2203.03605) is a state of the art transformer-based object detection model. Similar to Deformable DETR, DINO does not use heuristics based methods like NMS or IOU assignment found in convolution-based object detection models like Faster RCNN. Compared to Deformable DETR, DINO uses de-noising during training which can help training to converge faster.


![image](https://raw.githubusercontent.com/vpraveen-nv/model_card_images/main/api/automl_workflow.png)

---

### Sample prediction of a trained RT-DETR model

<img align="center" src="sample_images/detection_sample.jpg" width="640">

---

### Expected outcome
The expected output of this notebook is:
- A fully trained DINO object detection model optimized through AutoML
- Performance metrics comparing different hyperparameter configurations
- Exported model files ready for deployment in production environments
- Complete workflow documentation for airgapped deployments

Detailed documentation about DINO object detection is available in the TAO [documentation](https://docs.nvidia.com/tao/tao-toolkit/text/object_detection/dino.html)

---

### The workflow in a nutshell
This notebook demonstrates how to train DINO object detection models using AutoML in a completely local environment suitable for airgapped deployments:

1. **Setup Local Environment** - Configure local storage paths and verify prerequisites
2. **Connect to Local TAO API** - Connect to locally deployed TAO API service
3. **Create Local Workspace** - Setup local workspace without cloud dependencies
4. **Prepare Local Dataset** - Load and register datasets from local storage
5. **Configure DINO Model** - Setup DINO architecture and pretrained models
6. **Setup AutoML** - Configure automated hyperparameter optimization
7. **Train with AutoML** - Execute automated training with multiple trials
8. **Model Evaluation** - Evaluate best performing model
9. **Model Export** - Export optimized model for deployment
10. **Local Inference** - Run inference using local model files


---

### Requirements
Prior to running this notebook you must have:
1. A locally deployed TAO API server [(Local Setup Guide)](https://docs.nvidia.com/tao/tao-toolkit/text/tao_toolkit_api/api_setup.html#local-deployment)
2. Local object detection dataset in KITTI or COCO format stored in `~/data/` directory
3. Pretrained DINO models downloaded locally at ~/airgapped_models
4. Set the `<>` enclosed variables with values in the Configuration section
5. Sufficient local storage space (>50GB recommended for datasets and models)

### Airgapped Environment Setup
For completely airgapped environments, ensure the following are pre-downloaded:
- TAO API Docker images
- DINO pretrained model weights
- Sample datasets for testing


---

### Debugging Finetuning Microservice and Jobs

When working with the TAO API, you may encounter issues at different stages. Use the following guidance to debug effectively:

#### 1. Dataset, Experiment, or Workspace CRUD Operation Errors

If you encounter errors related to creating, reading, updating, or deleting datasets, experiments, or workspaces **and the error messages are not clear**, check the logs of the TAO API service pods:

```bash
docker logs -f tao_api_app
```

#### 2. Errors During Job Launch

For issues that occur **while launching a job**, check both the app and workflow pods:

```bash
docker logs -f tao_api_app
docker logs -f tao_api_workflow
```

#### 3. Errors After Job Launch

If errors occur **after a job has been launched**, inspect the job pod logs:

```bash
docker logs -f <job_id>
```

> **Note:**  
> Run these `docker` commands on the machine where your docker compose service is deployed.

#### Additional Debugging Tips

- **Job logs are automatically uploaded** to your cloud workspace at:  
  `/results/<job_id>/microservices_log.txt`

- **You can also view logs via the Jobs API endpoint:**  
  ```
  /api/v2/orgs/<org_name>/jobs/<job_id>/logs
  ```

**Summary Table**

| Error Type                | Where to Check Logs                                      |
|-------------------------- |---------------------------------------------------------|
| CRUD operation errors     | `tao-api-app-pod`                                       |
| Job launch errors         | `tao-api-app-pod`, `tao-api-workflow-pod`               |
| Post-launch job errors    | `tao-api-sts-<job_id>-0`                                |
| All job logs (cloud)      | `/results/<job_id>/microservices_log.txt`               |
| All job logs (API)        | `/api/v2/orgs/<org_name>/jobs/<job_id>/logs` |

---
### Performance Benchmarks

#### Execution Time Breakdown

The following table shows the approximate time required for each stage of the DINO AutoML FTMS workflow:

| **Stage** | **Duration** | **Description** |
|-----------|--------------|-----------------|
| Train Dataset Pull | **5s** | Train dataset verification and preprocessing |
| Val/Test Dataset Pull | **5s** | Validation and test dataset verification |
| AutoML Model Training | **1.5hrs** | 10 minutes per experiment |
| ONNX + TensorRT Export | **3min + 7min** | Model optimization and engine generation |
| TensorRT Inference | **20min** | High-performance inference testing |
| **Total Time** | **9hr 45min** | **Complete end-to-end workflow** |

#### Test Environment Specifications

| **Component** | **Specification** |
|---------------|-------------------|
| **GPU** | 1x NVIDIA A40 |
| **Training Dataset** | 50 images (5 MB total) |
| **Validation Dataset** | 50 images (5 MB total) |
| **Training Epochs** | 10 epochs per experiment |
| **Model Architecture** | Dino with Fan Tiny |
| **Task** | Warehouse object detection (4 classes) |

#### Performance Factors

> **Important Note:**  
> Actual execution times may vary significantly based on:
> 
> - **Hardware Configuration**: GPU type, memory, and compute capability
> - **Storage Performance**: Local disk I/O speed and cloud storage latency
> - **Network Conditions**: Bandwidth and latency to cloud storage
> - **System Load**: Other concurrent processes and resource utilization
> - **Dataset Size**: Number of images and total data volume
> - **Batch Size**: Larger batch sizes can improve training stability and speed
> - **Model Configuration**: Backbone architecture, epochs, AutoML configurations, and hyperparameters

## Transfer Datasets and Pre-trained models into local S3 bucket

### Local Dataset Structure
To see the dataset folder structure required for DINO object detection in local storage, ensure your data follows this pattern:

```
~/data/dino_train_dataset/
├── images.tar.gz
├── annotations.json
├── label_map.txt

~/data/dino_val_dataset/
├── images.tar.gz
├── annotations.json
├── label_map.txt
```

#### Setup AWS credentials as environment variables

In [ ]:
import os
# For minikube environment, uncomment this section until before docker compose section
# import subprocess

# # Get the Minikube IP using subprocess
# minikube_ip = subprocess.check_output(["minikube", "ip"]).decode("utf-8").strip()

# # Set environment variables
# os.environ["CLUSTER_IP"] = minikube_ip
# os.environ["SEAWEED_ENDPOINT"] = f"http://{os.environ["CLUSTER_IP"]}:32333"

# For docker compose environment, uncomment this section
# %env CLUSTER_IP=<MACHINE IP>
# %env SEAWEED_ENDPOINT=http://$CLUSTER_IP:8333
# os.environ["SEAWEED_ENDPOINT"] = f"http://{os.environ["CLUSTER_IP"]}:8333"


# Common for both
# Set AWS CLI credentials for SeaweedFS
%env AWS_ACCESS_KEY_ID=seaweedfs
%env AWS_SECRET_ACCESS_KEY=seaweedfs123
%env AWS_DEFAULT_REGION=us-east-1


#### Create a bucket if not present already

In [ ]:
%%bash
pip install awscli

# Create the main storage bucket if not already created
if ! aws s3 ls --endpoint-url "$SEAWEED_ENDPOINT" | grep -q "tao-storage"; then
    aws s3 mb --endpoint-url "$SEAWEED_ENDPOINT" s3://tao-storage
else
    echo "Bucket already exists, skipping creation."
fi

#### Copy data from local disk to local S3 bucket

In [ ]:
%%bash
aws s3 cp --endpoint-url $SEAWEED_ENDPOINT ~/data/dino_train_dataset s3://tao-storage/data/dino_train_dataset --recursive --quiet
aws s3 cp --endpoint-url $SEAWEED_ENDPOINT ~/data/dino_val_dataset s3://tao-storage/data/dino_val_dataset --recursive --quiet

# This directory that is being used on S3 to copied to is the one that should be present on LOCAL_MODEL_REGISTRY in values.yaml
aws s3 cp --endpoint-url $SEAWEED_ENDPOINT ~/airgapped-models/ s3://tao-storage/shared-storage/models/ --recursive --quiet

#### Verify if the uploads were successfull

In [ ]:
%%bash
aws s3 ls --endpoint-url $SEAWEED_ENDPOINT s3://tao-storage/shared-storage/models/
aws s3 ls --endpoint-url $SEAWEED_ENDPOINT s3://tao-storage/data/


## Configuration 

Fill in all `<>` enclosed variables with relevant values under this section. 

In [ ]:
import json
import os
import time
from IPython.display import clear_output

# Import TAO SDK
from tao_sdk.client import TaoClient

## Variable Restoration (for notebook session restart)

In [ ]:
# Restore variable in case of jupyter session restart and resume execution where it left off
%store -r workspace_id
%store -r job_map

## Configuration
Fill in all `<>` enclosed variables with relevant values under this section.

### TAO FTMS Host & Credentials

In [ ]:
# For Kubernetes: Use machine IP and port 32080
# For Docker Compose: Use localhost and port 8090 (or value set in config.env)
os.environ["TAO_BASE_URL"] = os.environ.get("TAO_BASE_URL", "http://localhost:8090/api/v2")

# Your NGC personal key
os.environ["NGC_KEY"] = ngc_key = os.environ.get("NGC_KEY", "<NGC_KEY>")

# Your NGC ORG name
os.environ["NGC_ORG"] = ngc_org_name = os.environ.get("NGC_ORG", "nvstaging")

# Docker environment variables (for private PTMs)
docker_env_vars = {}
# If using a PTM from a private organization like NVAIE, uncomment:
# docker_env_vars['TAO_API_KEY'] = '<NGC_LEGACY_API_KEY>'

### Cloud Storage Setup & Credentials (SeaweedFS for local deployment)

In [ ]:
# For local/airgapped deployments, use SeaweedFS as S3-compatible storage
cloud_metadata = {
    "name": "tao_workspace",
    "cloud_type": "seaweedfs",
    "cloud_specific_details": {
        "cloud_type": "seaweedfs",
        "cloud_region": "us-east-1",
        "cloud_bucket_name": "tao-storage",
        "access_key": "seaweedfs",
        "secret_key": "seaweedfs123",
        "endpoint_url": "http://seaweedfs-s3:8333"  # Internal Kubernetes service URL
    }
}

### Dataset Paths in Cloud Storage

In [ ]:
# Dataset paths: default to Seaweed (tao-storage) where we copied local data via aws s3 cp
os.environ["TRAIN_DATASET_URI"] = train_dataset_uri = os.environ.get("TRAIN_DATASET_URI", "seaweedfs://tao-storage/data/dino_train_dataset")
os.environ["EVAL_DATASET_URI"] = eval_dataset_uri = os.environ.get("EVAL_DATASET_URI", "seaweedfs://tao-storage/data/dino_val_dataset")

### Model Configuration

In [ ]:
# DINO Model Configuration
# Documentation: https://docs.nvidia.com/tao/tao-toolkit/text/object_detection/dino.html
model_name = "dino"
num_classes = 5

### AutoML Configuration

In [ ]:
automl_algorithm = "bayesian"  # bayesian or hyperband
automl_max_recommendations = 2  # Number of AutoML experiments to run


## Login

In [ ]:
# Initialize TAO Client and login using SDK
tao_client = TaoClient()

# Login using TAO SDK
login_response = tao_client.login(
    ngc_key=ngc_key,
    ngc_org_name=ngc_org_name,
    enable_telemetry=True
)

print("Login successful!")
print("API Base URL:", tao_client.base_url)
print("Organization:", tao_client.org_name)

## Create Cloud Workspace
For airgapped/local deployments, this creates a workspace pointing to SeaweedFS.

In [ ]:
# Create cloud workspace using TAO SDK
workspace_response = tao_client.create_workspace(
    name=cloud_metadata["name"],
    cloud_type=cloud_metadata["cloud_specific_details"].get("cloud_type", cloud_metadata.get("cloud_type")),
    cloud_specific_details=cloud_metadata["cloud_specific_details"]
)
workspace_id = workspace_response["id"]
if workspace_response.get("_duplicate"):
    print("ℹ️  Using existing workspace (already created)")
if workspace_response.get("_message"):
    print(f"ℹ️  {workspace_response['_message']}")

print("Workspace created successfully!")
print(f"Workspace ID: {workspace_id}")

# Get workspace details to confirm creation
workspace_details = tao_client.get_workspace_metadata(workspace_id)
print("Workspace details:")
print(json.dumps(workspace_details, indent=4))

## Load Base Experiments from Local Storage
For airgapped deployments, load PTMs from local storage into the database.

In [ ]:
# Load base experiments from local storage
# Requires: (1) the 'Copy data from local disk to local S3 bucket' cell ran (uploads ~/airgapped-models/ to s3://tao-storage/shared-storage/models/)
# (2) TAO API server has LOCAL_MODEL_REGISTRY=shared-storage/models (or /shared-storage/models) in its environment
try:
    airgapped_result = tao_client.load_airgapped_model(model_data={"workspace_id": workspace_id})
    print("Airgapped experiments loaded successfully!")
    print(json.dumps(airgapped_result, indent=4))
except Exception as e:
    print(f"Note: load_airgapped_experiments may not be available in all SDK versions: {e}")
    print("Continuing with PTM from database if already loaded...")

## Job Map and Common Parameters

In [ ]:
job_map = {}
encode_key = "tlt_encode"
checkpoint_choose_method = "best_model"


## Assign Pretrained Model
For airgapped deployments, PTMs should be loaded from local storage first.

In [ ]:
# List base experiments (PTMs) using TAO SDK
base_experiments = tao_client.list_base_experiments(filter_params={"network_arch": model_name})

print(f"Available base experiments (PTMs) for {model_name}:")
for exp in base_experiments:
    exp_name = exp.get("name", "N/A")
    exp_version = exp.get("version", "N/A")
    ngc_path = exp.get("ngc_path", "N/A")
    print(f"PTM Name: {exp_name}; PTM version: {exp_version}; NGC PATH: {ngc_path}")

In [ ]:
# Select PTM
pretrained_map = {"dino": "pretrained_dino_imagenet:fan_hybrid_tiny"}

selected_ptm_id = None
base_experiments_detailed = tao_client.list_base_experiments(filter_params={"network_arch": model_name})

for exp in base_experiments_detailed:
    ngc_path = exp.get("ngc_path", "")
    if ngc_path.endswith(pretrained_map[model_name]):
        selected_ptm_id = exp.get("id")
        print("Selected PTM metadata:")
        print(json.dumps(exp, indent=4))
        break

if not selected_ptm_id:
    print(f"PTM with NGC path ending in '{pretrained_map[model_name]}' not found!")
    print("Make sure the airgapped PTMs have been loaded correctly.")

## View AutoML Default Parameters

In [ ]:
# Get default AutoML parameters using TAO SDK
automl_params = tao_client.get_automl_defaults(network_arch=model_name, action="train")
print("Default AutoML parameters:")
print(json.dumps(automl_params, sort_keys=True, indent=4))


## Configure AutoML

In [ ]:
# Prepare algorithm-specific parameters based on the algorithm
algorithm_specific_params = {}
if automl_algorithm in ("bayesian", "bfbo"):
    algorithm_specific_params = {
        "automl_max_recommendations": 20  # Number of recommendations for Bayesian optimization
    }
elif automl_algorithm == "hyperband":
    algorithm_specific_params = {
        "automl_max_epochs": 27,  # Maximum epochs (was automl_R)
        "automl_reduction_factor": 3,  # Reduction factor (was automl_nu)
        "epoch_multiplier": 1  # Epoch multiplier for scaling
    }

# Refer to parameter list mentioned in the above links and add/remove any extra parameter in addition to the default enabled ones in automl_specs
automl_information = {
    "automl_enabled": True,
    "automl_algorithm": automl_algorithm,
    "automl_delete_intermediate_ckpt": False,
    "override_automl_disabled_params": False,
    "automl_hyperparameters": str(automl_params),
    "algorithm_specific_params": algorithm_specific_params,
    "metric": "kpi"
}

print("AutoML configuration prepared for job creation:")
print(json.dumps(automl_information, sort_keys=True, indent=4))
print("This will be included in the job_run_experiment call")

## Run AutoML Training

### Retrieve Default Training Spec

In [ ]:
# Get default train specs using TAO SDK
train_spec_response = tao_client.get_job_schema(action="train", network_arch=model_name)
train_specs = train_spec_response.get("default", {})
print("Default train specifications:")
print(json.dumps(train_specs, sort_keys=True, indent=4))


### Configure Training Spec

In [ ]:
# Customize train model specs
train_specs["train"]["num_epochs"] = 10
train_specs["train"]["checkpoint_interval"] = 10
train_specs["train"]["validation_interval"] = 10
train_specs["train"]["num_gpus"] = 1
train_specs["dataset"]["num_classes"] = num_classes

print(json.dumps(train_specs, sort_keys=True, indent=4))

### Submit AutoML Training Job

In [ ]:
job_name = f"{model_name}_automl_train_job"

train_job_response = tao_client.create_job(
    kind="experiment",
    name=job_name,
    network_arch=model_name,
    encryption_key=encode_key,
    workspace=workspace_id,
    train_dataset_uris=[train_dataset_uri],
    eval_dataset_uri=eval_dataset_uri,
    inference_dataset_uri=eval_dataset_uri,
    action="train",
    specs=train_specs,
    base_experiment_ids=[selected_ptm_id] if selected_ptm_id else None,
    automl_settings=automl_information,
    docker_env_vars=docker_env_vars,
)
train_job_id = train_job_response["id"]
if train_job_response.get("_duplicate"):
    print("ℹ️  Using existing job (already created)")
if train_job_response.get("_message"):
    print(f"ℹ️  {train_job_response['_message']}")

print("AutoML training job created successfully!")
print(f"Job ID: {train_job_id}")
print(f"Action: train (with AutoML)")

job_map["train_" + model_name] = train_job_id
print("\nJob Map:")
print(json.dumps(job_map, indent=4))

In [ ]:
# Monitor AutoML training job status
job_id = job_map["train_" + model_name]

while True:
    clear_output(wait=True)

    try:
        job_status = tao_client.get_job_metadata(job_id)

        print(f"AutoML Training Job Status")
        print(f"Job ID: {job_id}")
        print(f"Status: {job_status.get('status', 'Unknown')}")
        print(f"Progress: {job_status.get('progress', 'N/A')}")

        print("\nDetailed Status:")
        print(json.dumps(job_status.get("job_details", {}), sort_keys=True, indent=4))

        current_status = job_status.get("status", "Unknown")

        if current_status == "Error":
            raise Exception("AutoML training job failed!")

        if current_status in ["Done", "Completed"]:
            print("AutoML training job completed successfully!")
            break

        if current_status in ["Canceled", "Paused"]:
            print(f"Job {current_status}")
            break

    except Exception as e:
        if "failed" in str(e).lower():
            raise
        print(f"Error fetching job status: {str(e)}")
        print("Job might still be starting up...")

    time.sleep(15)

## Evaluate AutoML Model

In [ ]:
# Get default eval specs using TAO SDK
eval_spec_response = tao_client.get_job_schema(action="evaluate", network_arch=model_name)
eval_specs = eval_spec_response.get("default", {})
print("Default evaluate specifications:")
print(json.dumps(eval_specs, sort_keys=True, indent=4))

In [ ]:
# Customize Evaluation Spec
eval_specs["dataset"]["num_classes"] = num_classes
print(json.dumps(eval_specs, sort_keys=True, indent=4))

In [ ]:
# Submit Evaluation Job
parent_job_id = job_map["train_" + model_name]
eval_job_name = f"{model_name}_automl_eval_job"

eval_job_response = tao_client.create_job(
    kind="experiment",
    name=eval_job_name,
    network_arch=model_name,
    encryption_key=encode_key,
    workspace=workspace_id,
    eval_dataset_uri=eval_dataset_uri,
    action="evaluate",
    specs=eval_specs,
    parent_job_id=parent_job_id,
)
eval_job_id = eval_job_response["id"]
if eval_job_response.get("_duplicate"):
    print("ℹ️  Using existing job (already created)")
if eval_job_response.get("_message"):
    print(f"ℹ️  {eval_job_response['_message']}")

print("Evaluate job created successfully!")
print(f"Evaluate Job ID: {eval_job_id}")
print(f"Parent Job ID: {parent_job_id}")
print(f"Action: evaluate")

job_map["evaluate_" + model_name] = eval_job_id
print("\nUpdated Job Map:")
print(json.dumps(job_map, indent=4))

In [ ]:
# Monitor evaluate job status
job_id = job_map["evaluate_" + model_name]

while True:
    clear_output(wait=True)

    try:
        job_status = tao_client.get_job_metadata(job_id)

        print(f"Evaluate Job Status")
        print(f"Job ID: {job_id}")
        print(f"Status: {job_status.get('status', 'Unknown')}")
        print(f"Progress: {job_status.get('progress', 'N/A')}")

        print("\nDetailed Status:")
        print(json.dumps(job_status.get("job_details", {}), sort_keys=True, indent=4))

        current_status = job_status.get("status", "Unknown")

        if current_status == "Error":
            raise Exception("Evaluate job failed!")

        if current_status in ["Done", "Completed"]:
            print("Evaluate job completed successfully!")
            break

        if current_status in ["Canceled", "Paused"]:
            print(f"Job {current_status}")
            break

    except Exception as e:
        if "failed" in str(e).lower():
            raise
        print(f"Error fetching job status: {str(e)}")
        print("Job might still be starting up...")

    time.sleep(15)

## TRT Engine Generation
Export the AutoML model to ONNX format then convert to TensorRT engine.

### Export Model to ONNX

In [ ]:
# Get default export specs using TAO SDK
export_spec_response = tao_client.get_job_schema(action="export", network_arch=model_name)
export_specs = export_spec_response.get("default", {})
print("Default export specifications:")
print(json.dumps(export_specs, sort_keys=True, indent=4))

In [ ]:
# Customize export specs
export_specs["dataset"]["num_classes"] = num_classes
print(json.dumps(export_specs, sort_keys=True, indent=4))

In [ ]:
# Submit export job
parent_job_id = job_map["train_" + model_name]
export_job_name = f"{model_name}_automl_export_job"

export_job_response = tao_client.create_job(
    kind="experiment",
    name=export_job_name,
    network_arch=model_name,
    encryption_key=encode_key,
    workspace=workspace_id,
    action="export",
    specs=export_specs,
    parent_job_id=parent_job_id,
)
export_job_id = export_job_response["id"]
if export_job_response.get("_duplicate"):
    print("ℹ️  Using existing job (already created)")
if export_job_response.get("_message"):
    print(f"ℹ️  {export_job_response['_message']}")

print("Export job created successfully!")
print(f"Export Job ID: {export_job_id}")
print(f"Parent Job ID: {parent_job_id}")
print(f"Action: export")

job_map["export_" + model_name] = export_job_id
print("\nUpdated Job Map:")
print(json.dumps(job_map, indent=4))

In [ ]:
# Monitor export job status
job_id = job_map["export_" + model_name]

while True:
    clear_output(wait=True)

    try:
        job_status = tao_client.get_job_metadata(job_id)

        print(f"Export Job Status")
        print(f"Job ID: {job_id}")
        print(f"Status: {job_status.get('status', 'Unknown')}")
        print(f"Progress: {job_status.get('progress', 'N/A')}")

        print("\nDetailed Status:")
        print(json.dumps(job_status.get("job_details", {}), sort_keys=True, indent=4))

        current_status = job_status.get("status", "Unknown")

        if current_status == "Error":
            raise Exception("Export job failed!")

        if current_status in ["Done", "Completed"]:
            print("Export job completed successfully!")
            break

        if current_status in ["Canceled", "Paused"]:
            print(f"Job {current_status}")
            break

    except Exception as e:
        if "failed" in str(e).lower():
            raise
        print(f"Error fetching job status: {str(e)}")
        print("Job might still be starting up...")

    time.sleep(15)

### Convert ONNX to TRT Engine

In [ ]:
# Get default gen_trt_engine specs using TAO SDK
trt_spec_response = tao_client.get_job_schema(action="gen_trt_engine", network_arch=model_name)
trt_specs = trt_spec_response.get("default", {})
print("Default gen_trt_engine specifications:")
print(json.dumps(trt_specs, sort_keys=True, indent=4))

In [ ]:
# Customize gen_trt_engine specs
trt_specs["dataset"]["num_classes"] = num_classes
trt_specs["gen_trt_engine"]["tensorrt"]["data_type"] = "FP16"

In [ ]:
# Submit gen_trt_engine job
parent_job_id = job_map["export_" + model_name]
trt_job_name = f"{model_name}_gen_trt_engine_job"

trt_job_response = tao_client.create_job(
    kind="experiment",
    name=trt_job_name,
    network_arch=model_name,
    encryption_key=encode_key,
    workspace=workspace_id,
    action="gen_trt_engine",
    specs=trt_specs,
    parent_job_id=parent_job_id,
)
trt_job_id = trt_job_response["id"]
if trt_job_response.get("_duplicate"):
    print("ℹ️  Using existing job (already created)")
if trt_job_response.get("_message"):
    print(f"ℹ️  {trt_job_response['_message']}")

print("Generate TRT engine job created successfully!")
print(f"TRT Engine Job ID: {trt_job_id}")
print(f"Parent Job ID: {parent_job_id}")
print(f"Action: gen_trt_engine")

job_map["model_gen_trt_engine_" + model_name] = trt_job_id
print("\nUpdated Job Map:")
print(json.dumps(job_map, indent=4))

In [ ]:
# Monitor gen_trt_engine job status
job_id = job_map["model_gen_trt_engine_" + model_name]

while True:
    clear_output(wait=True)

    try:
        job_status = tao_client.get_job_metadata(job_id)

        print(f"Generate TRT Engine Job Status")
        print(f"Job ID: {job_id}")
        print(f"Status: {job_status.get('status', 'Unknown')}")
        print(f"Progress: {job_status.get('progress', 'N/A')}")

        print("\nDetailed Status:")
        print(json.dumps(job_status.get("job_details", {}), sort_keys=True, indent=4))

        current_status = job_status.get("status", "Unknown")

        if current_status == "Error":
            raise Exception("Generate TRT engine job failed!")

        if current_status in ["Done", "Completed"]:
            print("Generate TRT engine job completed successfully!")
            break

        if current_status in ["Canceled", "Paused"]:
            print(f"Job {current_status}")
            break

    except Exception as e:
        if "failed" in str(e).lower():
            raise
        print(f"Error fetching job status: {str(e)}")
        print("Job might still be starting up...")

    time.sleep(15)

### Inference TRT Engine

In [ ]:
# Get default inference specs using TAO SDK
inference_spec_response = tao_client.get_job_schema(action="inference", network_arch=model_name)
inference_specs = inference_spec_response.get("default", {})
print("Default inference specifications:")
print(json.dumps(inference_specs, sort_keys=True, indent=4))

In [ ]:
# Customize inference specs
inference_specs["dataset"]["num_classes"] = num_classes
inference_specs["dataset"]["batch_size"] = 1

In [ ]:
# Submit inference job
parent_job_id = job_map["model_gen_trt_engine_" + model_name]
inference_job_name = f"{model_name}_trt_inference_job"

inference_job_response = tao_client.create_job(
    kind="experiment",
    name=inference_job_name,
    network_arch=model_name,
    encryption_key=encode_key,
    workspace=workspace_id,
    inference_dataset_uri=eval_dataset_uri,
    action="inference",
    specs=inference_specs,
    parent_job_id=parent_job_id,
)
inference_job_id = inference_job_response["id"]
if inference_job_response.get("_duplicate"):
    print("ℹ️  Using existing job (already created)")
if inference_job_response.get("_message"):
    print(f"ℹ️  {inference_job_response['_message']}")

print("TRT Inference job created successfully!")
print(f"Inference Job ID: {inference_job_id}")
print(f"Parent Job ID: {parent_job_id}")
print(f"Action: inference")

job_map["inference_trt_" + model_name] = inference_job_id
print("\nUpdated Job Map:")
print(json.dumps(job_map, indent=4))

In [ ]:
# Monitor inference job status
job_id = job_map["inference_trt_" + model_name]

while True:
    clear_output(wait=True)

    try:
        job_status = tao_client.get_job_metadata(job_id)

        print(f"TRT Inference Job Status")
        print(f"Job ID: {job_id}")
        print(f"Status: {job_status.get('status', 'Unknown')}")
        print(f"Progress: {job_status.get('progress', 'N/A')}")

        print("\nDetailed Status:")
        print(json.dumps(job_status.get("job_details", {}), sort_keys=True, indent=4))

        current_status = job_status.get("status", "Unknown")

        if current_status == "Error":
            raise Exception("TRT Inference job failed!")

        if current_status in ["Done", "Completed"]:
            print("TRT Inference job completed successfully!")
            break

        if current_status in ["Canceled", "Paused"]:
            print(f"Job {current_status}")
            break

    except Exception as e:
        if "failed" in str(e).lower():
            raise
        print(f"Error fetching job status: {str(e)}")
        print("Job might still be starting up...")

    time.sleep(15)

## Clean Up
You can optionally run this section to delete the experiment results.

### Delete Jobs

In [ ]:
# Delete all created jobs using TAO SDK
print("Deleting all created jobs...")

for job_key, job_id in job_map.items():
    try:
        delete_result = tao_client.delete_job(job_id)
        print(f"Deleted job: {job_key} (ID: {job_id})")
    except Exception as e:
        print(f"Failed to delete job {job_key} (ID: {job_id}): {str(e)}")

print(f"\nJob cleanup completed!")